In [1]:
import numpy as np
import pandas as pd
import rebound
from rebound import Simulation, Particle
import torch

In [2]:
from sbi.inference import NPE
from sbi import utils as sbi_utils

/work/phillip/anaconda3/envs/sbi_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from sbi import utils as sbi_utils
from sbi.inference import NPE, simulate_for_sbi
from sbi.utils import BoxUniform

In [4]:
sim = rebound.Simulation()
sim.units = ('AU', 'yr', 'Msun')
sim.add(m=1)
sim.add(m=2e-3, a=1.)
sim.add(m=1e-2, a=2., e=0.1)
sim.add(m=1e-3, a=1.5, e=0.2)
sim.integrate(10000.)

In [5]:
for p in sim.particles:
    print(p.x, p.y, p.z)
for o in sim.orbits():
    print(o)

9.069481221501414 686.4457888782873 0.0
9.563185496997619 687.4523442227804 0.0
9.344682615920052 688.4220100675009 0.0
-9160.814676004733 -9702.947327981963 0.0
<rebound.Orbit instance, a=0.880060961055079 e=0.33012296683816766 inc=0.0 Omega=0.0 omega=4.682798950859981 f=2.7151728221963722>
<rebound.Orbit instance, a=1.8310020620049035 e=0.11281535219582094 inc=0.0 Omega=0.0 omega=3.9734463613276505 f=3.7425193934890437>
<rebound.Orbit instance, a=-20.681391667269533 e=1.0967047949133608 inc=0.0 Omega=0.0 omega=1.2714346693967347 f=2.7178257573988844>


In [6]:
from rebound_simulator import simulator_for_sbi, simulator_single, simulate, summary_statistics

In [7]:
prior = sbi_utils.BoxUniform(
    low=torch.tensor([0.1, 0.1, 0.1, 0.1, 0, 0.1, 0, 0.1, 0, 0.1, 0, 0.1, 0, 0.1, 0]),
    high=torch.tensor([10, 10, 10, 5, 10, 5, 10, 5, 10, 5, 10, 5, 10, 5, 10])
)

In [8]:
from joblib import Parallel, delayed
from tqdm import tqdm

In [ ]:
all_params = prior.sample((100000,)).numpy()
results = Parallel(n_jobs=10, backend="multiprocessing", verbose=10)(
    delayed(simulator_single)(all_params[i]) for i in tqdm(range(100000))
)

theta = torch.tensor(all_params, dtype=torch.float32)
x = torch.from_numpy(np.stack(results).astype(np.float32))


  6%|▌         | 5959/100000 [01:41<26:37, 58.85it/s]
[Parallel(n_jobs=10)]: Using backend MultiprocessingBackend with 10 concurrent workers.

[Parallel(n_jobs=10)]: Batch computation too fast (0.0043277740478515625s.) Setting batch_size=2.
[Parallel(n_jobs=10)]: Done   5 tasks      | elapsed:    0.3s
[Parallel(n_jobs=10)]: Done  12 tasks      | elapsed:    0.3s
[Parallel(n_jobs=10)]: Batch computation too fast (0.019391536712646484s.) Setting batch_size=4.
[Parallel(n_jobs=10)]: Done  26 tasks      | elapsed:    0.3s
[Parallel(n_jobs=10)]: Done  43 tasks      | elapsed:    0.4s

100%|██████████| 100/100 [00:00<00:00, 223.77it/s]
[Parallel(n_jobs=10)]: Done  70 out of 100 | elapsed:    0.6s remaining:    0.3s
[Parallel(n_jobs=10)]: Done 100 out of 100 | elapsed:    1.1s finished


In [18]:
np.save("data/theta.npy", theta, allow_pickle=True)
np.save("data/x.npy", x, allow_pickle=True)

In [15]:
print(x.shape)
print(theta.shape)

torch.Size([100, 12])
torch.Size([100, 15])


In [ ]:
inference = NPE(prior=prior)
inference.append_simulations(theta, x)
density_estimator = inference.train()

/usr/local/lib/python3.12/dist-packages/sbi/inference/trainers/npe/npe_base.py:196: UserWarning: Data has extreme outliers in dimension(s) [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] (beyond 10.0x IQR from quartiles). This may cause precision loss during z-scoring, where distinct values become indistinguishable. Consider removing outliers from your data or setting `z_score_x='none'` (though this may affect training).
  warn_if_invalid_for_zscoring(x)


 Neural network successfully converged after 305 epochs.

In [ ]:
posterior = inference.build_posterior(density_estimator)

In [ ]:
x_obs = torch.tensor(summary_statistics(simulate(1, 1, 1, 0, 0, 1, 2, 3, 0.6, 0, 0, 1, 1, 0.9, 0.3)))

In [ ]:
posterior.set_default_x(x_obs)
map_estimate = posterior.map()

  0%|          | 0/1000 [00:00<?, ?it/s]

In [ ]:
print("MAP:", map_estimate)

MAP: tensor([[3.1165, 4.5727, 4.7304, 2.4400, 1.2839, 3.4376, 0.5381, 2.6478, 0.7759,
         2.4277, 0.7188, 2.6271, 0.1138, 2.6698, 0.5349]])
